In [26]:
from pathlib import Path
import sqlite3
import pandas as pd

# Project directories
PROJECT_ROOT = Path.cwd().parent
RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
SQL_DIR = PROJECT_ROOT / "sql"

# Create SQLite database
DB_PATH = PROJECT_ROOT / "data" / "finance_rtr.db"

connection = sqlite3.connect(DB_PATH)

print("SQL DATABASE INITIALISED")
print("=" * 45)
print("Project:", PROJECT_ROOT)
print("Database:", DB_PATH)
print("Database exists:", DB_PATH.exists())
print("SQLite connection:", connection is not None)

SQL DATABASE INITIALISED
Project: C:\Users\hp\Desktop\Gdansk University of Technology, Poland\Job\AON\Project\finance-rtr-analytics
Database: C:\Users\hp\Desktop\Gdansk University of Technology, Poland\Job\AON\Project\finance-rtr-analytics\data\finance_rtr.db
Database exists: True
SQLite connection: True


In [27]:
# Map CSV files to SQL table names
csv_tables = {
    "general_ledger": "general_ledger.csv",
    "chart_of_accounts": "chart_of_accounts.csv",
    "bank_statement": "bank_statement.csv",
    "accounts_payable": "accounts_payable.csv",
    "accounts_receivable": "accounts_receivable.csv",
    "budget": "budget.csv"
}

# Import each CSV into SQLite
for table_name, filename in csv_tables.items():
    file_path = RAW_DATA / filename

    df = pd.read_csv(file_path)

    df.to_sql(
        table_name,
        connection,
        if_exists="replace",
        index=False
    )

    print(f"{table_name}: {len(df)} records imported")

print("\nAll six financial datasets imported.")

general_ledger: 104 records imported
chart_of_accounts: 20 records imported
bank_statement: 26 records imported
accounts_payable: 12 records imported
accounts_receivable: 12 records imported
budget: 10 records imported

All six financial datasets imported.


In [28]:
# Retrieve all tables in the SQLite database
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection
)

print("DATABASE TABLES")
display(tables)

# Verify record counts
print("\nRECORD COUNT VERIFICATION")

for table_name in csv_tables.keys():
    count = pd.read_sql_query(
        f"SELECT COUNT(*) AS total FROM {table_name}",
        connection
    ).iloc[0, 0]

    print(f"{table_name:<25} {count:>5} records")

DATABASE TABLES


,name
0,accounts_payable
1,accounts_receivable
2,bank_statement
3,budget
4,chart_of_accounts
5,general_ledger



RECORD COUNT VERIFICATION
general_ledger              104 records
chart_of_accounts            20 records
bank_statement               26 records
accounts_payable             12 records
accounts_receivable          12 records
budget                       10 records


In [29]:
sql_financial_summary = """
SELECT
    coa.account_type,

    ROUND(
        SUM(
            CASE
                WHEN coa.normal_balance = 'Credit'
                    THEN gl.credit - gl.debit
                ELSE gl.debit - gl.credit
            END
        ), 2
    ) AS actual_amount

FROM general_ledger AS gl

INNER JOIN chart_of_accounts AS coa
    ON gl.account_code = coa.account_code

GROUP BY coa.account_type

ORDER BY coa.account_type;
"""

financial_summary_sql = pd.read_sql_query(
    sql_financial_summary,
    connection
)

display(financial_summary_sql)

,account_type,actual_amount
0,Asset,117997.99
1,Equity,50000.00
2,Expense,68915.59
3,Liability,34849.67
4,Revenue,101563.91


In [30]:
sql_financial_summary = """
SELECT
    coa.account_type,

    ROUND(
        SUM(
            CASE
                WHEN coa.account_type IN ('Asset', 'Expense')
                    THEN gl.debit - gl.credit
                ELSE gl.credit - gl.debit
            END
        ), 2
    ) AS actual_amount

FROM general_ledger AS gl

INNER JOIN chart_of_accounts AS coa
    ON gl.account_code = coa.account_code

GROUP BY coa.account_type

ORDER BY coa.account_type;
"""

financial_summary_sql = pd.read_sql_query(
    sql_financial_summary,
    connection
)

display(financial_summary_sql)

,account_type,actual_amount
0,Asset,117497.99
1,Equity,50000.00
2,Expense,68915.59
3,Liability,34849.67
4,Revenue,101563.91


In [31]:
sql_profit = """
SELECT
    ROUND(
        SUM(
            CASE
                WHEN coa.account_type = 'Revenue'
                    THEN gl.credit - gl.debit
                WHEN coa.account_type = 'Expense'
                    THEN -(gl.debit - gl.credit)
                ELSE 0
            END
        ), 2
    ) AS net_profit

FROM general_ledger AS gl

INNER JOIN chart_of_accounts AS coa
    ON gl.account_code = coa.account_code;
"""

profit_sql = pd.read_sql_query(sql_profit, connection)

sql_net_profit = profit_sql.loc[0, "net_profit"]

print("SQL NET PROFIT")
print("=" * 40)
print(f"Net Profit: €{sql_net_profit:,.2f}")

SQL NET PROFIT
Net Profit: €32,648.32


In [32]:
python_net_profit = 32648.32

difference = round(sql_net_profit - python_net_profit, 2)

print("SQL vs PYTHON VALIDATION")
print("=" * 40)
print(f"SQL Net Profit:    €{sql_net_profit:,.2f}")
print(f"Python Net Profit: €{python_net_profit:,.2f}")
print(f"Difference:        €{difference:,.2f}")
print("Status:", "PASS" if difference == 0 else "INVESTIGATE")

SQL vs PYTHON VALIDATION
SQL Net Profit:    €32,648.32
Python Net Profit: €32,648.32
Difference:        €0.00
Status: PASS


In [33]:
sql_account_balances = """
SELECT
    coa.account_code,
    coa.account_name,
    coa.account_type,

    ROUND(COALESCE(SUM(gl.debit), 0), 2) AS total_debit,
    ROUND(COALESCE(SUM(gl.credit), 0), 2) AS total_credit,

    ROUND(
        CASE
            WHEN coa.account_type IN ('Asset', 'Expense')
                THEN COALESCE(SUM(gl.debit), 0)
                     - COALESCE(SUM(gl.credit), 0)
            ELSE COALESCE(SUM(gl.credit), 0)
                 - COALESCE(SUM(gl.debit), 0)
        END,
        2
    ) AS closing_balance

FROM chart_of_accounts AS coa

LEFT JOIN general_ledger AS gl
    ON coa.account_code = gl.account_code

GROUP BY
    coa.account_code,
    coa.account_name,
    coa.account_type

ORDER BY coa.account_code;
"""

account_balances = pd.read_sql_query(
    sql_account_balances,
    connection
)

display(account_balances)

,account_code,account_name,account_type,total_debit,total_credit,closing_balance
0,1000,Bank Account,Asset,142075.75,62315.92,79759.83
1,1100,Accounts Receivable,Asset,101413.91,71925.75,29488.16
2,1200,Prepaid Expenses,Asset,0.00,0.00,0.00
3,1500,Equipment,Asset,8500.00,0.00,8500.00
4,1590,Accumulated Depreciation,Asset,0.00,250.00,-250.00
5,2000,Accounts Payable,Liability,25814.50,40664.17,14849.67
6,2100,Accrued Expenses,Liability,0.00,0.00,0.00
7,2200,Bank Loan,Liability,0.00,20000.00,20000.00
8,3000,Share Capital,Equity,0.00,50000.00,50000.00
9,3100,Retained Earnings,Equity,0.00,0.00,0.00


In [34]:
print("COMPLETE CHART OF ACCOUNTS REPORT")
print("=" * 45)

print("Accounts in report:", len(account_balances))
print("Accounts in Chart of Accounts:",
      pd.read_sql_query(
          "SELECT COUNT(*) AS total FROM chart_of_accounts",
          connection
      ).loc[0, "total"])

print("Accounts with zero activity:",
      (account_balances["total_debit"] == 0).sum()
      + 0) # Temporary indicator; see note below.

COMPLETE CHART OF ACCOUNTS REPORT
Accounts in report: 20
Accounts in Chart of Accounts: 20
Accounts with zero activity: 9


In [35]:
zero_activity = account_balances[
    (account_balances["total_debit"] == 0) &
    (account_balances["total_credit"] == 0)
]

print("ACCOUNT ACTIVITY VALIDATION")
print("=" * 45)

print("Total accounts:", len(account_balances))
print("Accounts with activity:",
      len(account_balances) - len(zero_activity))
print("Accounts with zero activity:", len(zero_activity))

print("\nINACTIVE ACCOUNTS")
display(
    zero_activity[
        ["account_code", "account_name", "account_type"]
    ]
)

ACCOUNT ACTIVITY VALIDATION
Total accounts: 20
Accounts with activity: 16
Accounts with zero activity: 4

INACTIVE ACCOUNTS


,account_code,account_name,account_type
2,1200,Prepaid Expenses,Asset
6,2100,Accrued Expenses,Liability
9,3100,Retained Earnings,Equity
19,5700,Interest Expense,Expense


In [36]:
print("GENERAL LEDGER COLUMNS")
print("=" * 45)

display(pd.read_sql_query(
    "PRAGMA table_info(general_ledger)",
    connection
))

print("\nBANK STATEMENT COLUMNS")
print("=" * 45)

display(pd.read_sql_query(
    "PRAGMA table_info(bank_statement)",
    connection
))

GENERAL LEDGER COLUMNS


,cid,name,type,notnull,dflt_value,pk
0,0,transaction_id,TEXT,0,None,0
1,1,date,TEXT,0,None,0
2,2,description,TEXT,0,None,0
3,3,department,TEXT,0,None,0
4,4,account_code,INTEGER,0,None,0
5,5,debit,REAL,0,None,0
6,6,credit,REAL,0,None,0



BANK STATEMENT COLUMNS


,cid,name,type,notnull,dflt_value,pk
0,0,bank_transaction_id,TEXT,0,None,0
1,1,transaction_id,TEXT,0,None,0
2,2,date,TEXT,0,None,0
3,3,description,TEXT,0,None,0
4,4,credit,REAL,0,None,0
5,5,debit,REAL,0,None,0
6,6,signed_amount,REAL,0,None,0


In [37]:
print("GENERAL LEDGER COLUMNS")
print("=" * 45)

display(pd.read_sql_query(
    "PRAGMA table_info(general_ledger)",
    connection
))

print("\nBANK STATEMENT COLUMNS")
print("=" * 45)

display(pd.read_sql_query(
    "PRAGMA table_info(bank_statement)",
    connection
))

GENERAL LEDGER COLUMNS


,cid,name,type,notnull,dflt_value,pk
0,0,transaction_id,TEXT,0,None,0
1,1,date,TEXT,0,None,0
2,2,description,TEXT,0,None,0
3,3,department,TEXT,0,None,0
4,4,account_code,INTEGER,0,None,0
5,5,debit,REAL,0,None,0
6,6,credit,REAL,0,None,0



BANK STATEMENT COLUMNS


,cid,name,type,notnull,dflt_value,pk
0,0,bank_transaction_id,TEXT,0,None,0
1,1,transaction_id,TEXT,0,None,0
2,2,date,TEXT,0,None,0
3,3,description,TEXT,0,None,0
4,4,credit,REAL,0,None,0
5,5,debit,REAL,0,None,0
6,6,signed_amount,REAL,0,None,0


In [38]:
sql_duplicate_bank = """
SELECT
    transaction_id,
    COUNT(*) AS record_count,
    ROUND(SUM(debit), 2) AS total_debit,
    ROUND(SUM(credit), 2) AS total_credit
FROM bank_statement
GROUP BY transaction_id
HAVING COUNT(*) > 1
ORDER BY transaction_id;
"""

duplicate_bank_records = pd.read_sql_query(
    sql_duplicate_bank,
    connection
)

print("DUPLICATE BANK TRANSACTIONS")
print("=" * 45)
display(duplicate_bank_records)

DUPLICATE BANK TRANSACTIONS


,transaction_id,record_count,total_debit,total_credit
0,TXN00018,2,0.0,22681.76


In [39]:
sql_gl_bank_movements = """
SELECT
    transaction_id,
    date,
    description,
    ROUND(debit, 2) AS debit,
    ROUND(credit, 2) AS credit,
    ROUND(debit - credit, 2) AS signed_amount
FROM general_ledger
WHERE account_code = 1000
ORDER BY date, transaction_id;
"""

gl_bank_movements = pd.read_sql_query(
    sql_gl_bank_movements,
    connection
)

print("GENERAL LEDGER BANK MOVEMENTS")
print("=" * 45)
print("Number of bank-account entries:", len(gl_bank_movements))
display(gl_bank_movements.head(10))

GENERAL LEDGER BANK MOVEMENTS
Number of bank-account entries: 27


,transaction_id,date,description,debit,credit,signed_amount
0,OPENING001,2026-11-01,Opening bank balance,50000.00,0.00,50000.00
1,TXN00041,2026-11-03 00:00:00,Monthly salaries,0.00,21391.42,-21391.42
2,TXN00043,2026-11-03 00:00:00,Utilities,0.00,932.15,-932.15
3,TXN00049,2026-11-05 00:00:00,Bank loan received,20000.00,0.00,20000.00
4,TXN00013,2026-11-08 00:00:00,Customer payment - Evergreen Retail Group,6949.91,0.00,6949.91
5,TXN00039,2026-11-08 00:00:00,Supplier payment - CloudTech Services,0.00,4926.20,-4926.20
6,TXN00045,2026-11-10 00:00:00,Business travel,0.00,733.90,-733.90
7,TXN00015,2026-11-11 00:00:00,Customer payment - Alpine Consulting,11780.60,0.00,11780.60
8,TXN00037,2026-11-12 00:00:00,Supplier payment - CloudTech Services,0.00,2904.34,-2904.34
9,TXN00047,2026-11-12 00:00:00,Purchase of office equipment,0.00,8500.00,-8500.00


In [40]:
sql_bank_sample = """
SELECT
    bank_transaction_id,
    transaction_id,
    date,
    description,
    debit,
    credit,
    signed_amount
FROM bank_statement
ORDER BY date, transaction_id
LIMIT 10;
"""

bank_sample = pd.read_sql_query(
    sql_bank_sample,
    connection
)

display(bank_sample)

,bank_transaction_id,transaction_id,date,description,debit,credit,signed_amount
0,BANK00017,TXN00041,2026-11-03,Monthly salaries,21391.42,0.00,-21391.42
1,BANK00019,TXN00043,2026-11-03,Utilities,932.15,0.00,-932.15
2,BANK00024,TXN00049,2026-11-05,Bank loan received,0.00,20000.00,20000.00
3,BANK00001,TXN00013,2026-11-08,Customer payment - Evergreen Retail Group,0.00,6949.91,6949.91
4,BANK00015,TXN00039,2026-11-08,Supplier payment - CloudTech Services,4926.20,0.00,-4926.20
5,BANK00021,TXN00045,2026-11-10,Business travel,733.90,0.00,-733.90
6,BANK00013,TXN00037,2026-11-12,Supplier payment - CloudTech Services,2904.34,0.00,-2904.34
7,BANK00023,TXN00047,2026-11-12,Purchase of office equipment,8500.00,0.00,-8500.00
8,BANK00012,TXN00036,2026-11-13,Supplier payment - City Properties,3116.38,0.00,-3116.38
9,BANK00022,TXN00046,2026-11-13,Bank charges,142.92,0.00,-142.92


In [41]:
sql_special_bank_records = """
SELECT *
FROM bank_statement
WHERE transaction_id IN ('TXN00015', 'TXN00018', 'BANKFEE001')
ORDER BY transaction_id, bank_transaction_id;
"""

special_bank_records = pd.read_sql_query(
    sql_special_bank_records,
    connection
)

display(special_bank_records)

,bank_transaction_id,transaction_id,date,description,credit,debit,signed_amount
0,BANK00027,BANKFEE001,2026-11-28,Unrecorded bank service fee,0.00,45.0,-45.00
1,BANK00006,TXN00018,2026-11-17,Customer payment - Central European Logistics,11340.88,0.0,11340.88
2,BANK00026,TXN00018,2026-11-17,Customer payment - Central European Logistics ...,11340.88,0.0,11340.88


In [42]:
sql_bank_reconciliation = """
WITH gl_bank AS (
    SELECT
        transaction_id,
        ROUND(SUM(debit - credit), 2) AS gl_signed_amount
    FROM general_ledger
    WHERE account_code = 1000
      AND transaction_id != 'OPENING001'
    GROUP BY transaction_id
),

bank AS (
    SELECT
        transaction_id,
        COUNT(*) AS bank_record_count,
        ROUND(SUM(signed_amount), 2) AS bank_signed_amount
    FROM bank_statement
    GROUP BY transaction_id
),

all_transactions AS (
    SELECT transaction_id FROM gl_bank
    UNION
    SELECT transaction_id FROM bank
)

SELECT
    t.transaction_id,
    g.gl_signed_amount,
    b.bank_signed_amount,
    COALESCE(b.bank_record_count, 0) AS bank_record_count,

    CASE
        WHEN b.bank_record_count > 1
            THEN 'Duplicate Bank Transaction'

        WHEN g.transaction_id IS NULL
            THEN 'Not Recorded in General Ledger'

        WHEN b.transaction_id IS NULL
            THEN 'Missing from Bank Statement'

        WHEN ABS(g.gl_signed_amount + b.bank_signed_amount) > 0.01
            THEN 'Amount Mismatch'

        ELSE 'Matched'
    END AS reconciliation_status

FROM all_transactions AS t

LEFT JOIN gl_bank AS g
    ON t.transaction_id = g.transaction_id

LEFT JOIN bank AS b
    ON t.transaction_id = b.transaction_id

ORDER BY t.transaction_id;
"""

sql_reconciliation = pd.read_sql_query(
    sql_bank_reconciliation,
    connection
)

display(sql_reconciliation)

,transaction_id,gl_signed_amount,bank_signed_amount,bank_record_count,reconciliation_status
0,BANKFEE001,-45.00,-45.00,1,Amount Mismatch
1,TXN00013,6949.91,6949.91,1,Amount Mismatch
2,TXN00014,9276.31,9276.31,1,Amount Mismatch
3,TXN00015,11780.60,NaN,0,Missing from Bank Statement
4,TXN00016,10074.58,10074.58,1,Amount Mismatch
5,TXN00017,7053.47,7053.47,1,Amount Mismatch
6,TXN00018,11340.88,22681.76,2,Duplicate Bank Transaction
7,TXN00019,10404.85,10404.85,1,Amount Mismatch
8,TXN00020,5045.15,5045.15,1,Amount Mismatch
9,TXN00033,-4596.19,-4596.19,1,Amount Mismatch


In [43]:
print("SQL BANK RECONCILIATION SUMMARY")
print("=" * 50)

status_summary = (
    sql_reconciliation["reconciliation_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="transaction_count")
)

display(status_summary)

print("\nTOTAL TRANSACTION REFERENCES:",
      len(sql_reconciliation))

print(
    "MATCHED TRANSACTIONS:",
    (sql_reconciliation["reconciliation_status"] == "Matched").sum()
)

print(
    "OPEN EXCEPTIONS:",
    (sql_reconciliation["reconciliation_status"] != "Matched").sum()
)

SQL BANK RECONCILIATION SUMMARY


,status,transaction_count
0,Amount Mismatch,24
1,Missing from Bank Statement,1
2,Duplicate Bank Transaction,1



TOTAL TRANSACTION REFERENCES: 26
MATCHED TRANSACTIONS: 0
OPEN EXCEPTIONS: 26


In [44]:
sql_bank_reconciliation = """
WITH gl_bank AS (
    SELECT
        transaction_id,
        ROUND(SUM(debit - credit), 2) AS gl_signed_amount
    FROM general_ledger
    WHERE account_code = 1000
      AND transaction_id != 'OPENING001'
    GROUP BY transaction_id
),

bank AS (
    SELECT
        transaction_id,
        COUNT(*) AS bank_record_count,
        ROUND(SUM(signed_amount), 2) AS bank_signed_amount
    FROM bank_statement
    GROUP BY transaction_id
),

all_transactions AS (
    SELECT transaction_id FROM gl_bank
    UNION
    SELECT transaction_id FROM bank
)

SELECT
    t.transaction_id,
    g.gl_signed_amount,
    b.bank_signed_amount,
    COALESCE(b.bank_record_count, 0) AS bank_record_count,

    CASE
        WHEN b.bank_record_count > 1
            THEN 'Duplicate Bank Transaction'

        WHEN g.transaction_id IS NULL
            THEN 'Not Recorded in General Ledger'

        WHEN b.transaction_id IS NULL
            THEN 'Missing from Bank Statement'

        WHEN ABS(g.gl_signed_amount - b.bank_signed_amount) > 0.01
            THEN 'Amount Mismatch'

        ELSE 'Matched'
    END AS reconciliation_status

FROM all_transactions AS t

LEFT JOIN gl_bank AS g
    ON t.transaction_id = g.transaction_id

LEFT JOIN bank AS b
    ON t.transaction_id = b.transaction_id

ORDER BY t.transaction_id;
"""

sql_reconciliation = pd.read_sql_query(
    sql_bank_reconciliation,
    connection
)

display(sql_reconciliation)

,transaction_id,gl_signed_amount,bank_signed_amount,bank_record_count,reconciliation_status
0,BANKFEE001,-45.00,-45.00,1,Matched
1,TXN00013,6949.91,6949.91,1,Matched
2,TXN00014,9276.31,9276.31,1,Matched
3,TXN00015,11780.60,NaN,0,Missing from Bank Statement
4,TXN00016,10074.58,10074.58,1,Matched
5,TXN00017,7053.47,7053.47,1,Matched
6,TXN00018,11340.88,22681.76,2,Duplicate Bank Transaction
7,TXN00019,10404.85,10404.85,1,Matched
8,TXN00020,5045.15,5045.15,1,Matched
9,TXN00033,-4596.19,-4596.19,1,Matched


In [45]:
print("SQL BANK RECONCILIATION SUMMARY")
print("=" * 50)

status_summary = (
    sql_reconciliation["reconciliation_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="transaction_count")
)

display(status_summary)

print("\nTOTAL TRANSACTION REFERENCES:",
      len(sql_reconciliation))

print(
    "MATCHED TRANSACTIONS:",
    (sql_reconciliation["reconciliation_status"] == "Matched").sum()
)

print(
    "OPEN EXCEPTIONS:",
    (sql_reconciliation["reconciliation_status"] != "Matched").sum()
)

SQL BANK RECONCILIATION SUMMARY


,status,transaction_count
0,Matched,24
1,Missing from Bank Statement,1
2,Duplicate Bank Transaction,1



TOTAL TRANSACTION REFERENCES: 26
MATCHED TRANSACTIONS: 24
OPEN EXCEPTIONS: 2


In [46]:
print("SQL BANK RECONCILIATION SUMMARY")
print("=" * 50)

status_summary = (
    sql_reconciliation["reconciliation_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="transaction_count")
)

display(status_summary)

print("\nTOTAL TRANSACTION REFERENCES:",
      len(sql_reconciliation))

print(
    "MATCHED TRANSACTIONS:",
    (sql_reconciliation["reconciliation_status"] == "Matched").sum()
)

print(
    "OPEN EXCEPTIONS:",
    (sql_reconciliation["reconciliation_status"] != "Matched").sum()
)

SQL BANK RECONCILIATION SUMMARY


,status,transaction_count
0,Matched,24
1,Missing from Bank Statement,1
2,Duplicate Bank Transaction,1



TOTAL TRANSACTION REFERENCES: 26
MATCHED TRANSACTIONS: 24
OPEN EXCEPTIONS: 2


In [48]:
python_reconciliation_path = (
    PROCESSED_DATA / "bank_reconciliation.csv"
)

python_reconciliation = pd.read_csv(
    python_reconciliation_path
)

print("Python reconciliation loaded successfully.")
print("Records:", len(python_reconciliation))
print("\nAvailable columns:")
print(python_reconciliation.columns.tolist())

display(python_reconciliation.head())

Python reconciliation loaded successfully.
Records: 26

Available columns:
['transaction_id', 'description', 'gl_amount', 'bank_transaction_id', 'bank_amount', '_merge', 'reconciliation_status', 'duplicate_flag']


,transaction_id,description,gl_amount,bank_transaction_id,bank_amount,_merge,reconciliation_status,duplicate_flag
0,BANKFEE001,Record bank service fee,-45.00,BANK00027,-45.00,both,Matched,False
1,TXN00013,Customer payment - Evergreen Retail Group,6949.91,BANK00001,6949.91,both,Matched,False
2,TXN00014,Customer payment - FutureWorks Ltd.,9276.31,BANK00002,9276.31,both,Matched,False
3,TXN00015,Customer payment - Alpine Consulting,11780.60,NaN,NaN,left_only,Missing from Bank Statement,False
4,TXN00016,Customer payment - Evergreen Retail Group,10074.58,BANK00004,10074.58,both,Matched,False


In [49]:
print(
    python_reconciliation[
        "reconciliation_status"
    ].value_counts()
)

reconciliation_status
Matched                        25
Missing from Bank Statement     1
Name: count, dtype: int64


In [50]:
print("PYTHON DUPLICATE TRANSACTION CHECK")
print("=" * 45)

print(
    "Records flagged as duplicates:",
    python_reconciliation["duplicate_flag"].sum()
)

display(
    python_reconciliation[
        python_reconciliation["duplicate_flag"] == True
    ]
)

PYTHON DUPLICATE TRANSACTION CHECK
Records flagged as duplicates: 1


,transaction_id,description,gl_amount,bank_transaction_id,bank_amount,_merge,reconciliation_status,duplicate_flag
6,TXN00018,Customer payment - Central European Logistics,11340.88,BANK00006,11340.88,both,Matched,True


In [51]:
python_reconciliation["standard_status"] = (
    python_reconciliation["reconciliation_status"]
)

python_reconciliation.loc[
    python_reconciliation["duplicate_flag"] == True,
    "standard_status"
] = "Duplicate Bank Transaction"

python_status_counts = (
    python_reconciliation["standard_status"]
    .value_counts()
    .to_dict()
)

sql_status_counts = (
    sql_reconciliation["reconciliation_status"]
    .value_counts()
    .to_dict()
)

all_statuses = (
    set(python_status_counts) | set(sql_status_counts)
)

comparison = pd.DataFrame([
    {
        "status": status,
        "python_count": python_status_counts.get(status, 0),
        "sql_count": sql_status_counts.get(status, 0),
        "result": (
            "PASS"
            if python_status_counts.get(status, 0)
            == sql_status_counts.get(status, 0)
            else "INVESTIGATE"
        )
    }
    for status in sorted(all_statuses)
])

display(comparison)

print(
    "\nOVERALL VALIDATION:",
    "PASS" if (comparison["result"] == "PASS").all()
    else "INVESTIGATE"
)

,status,python_count,sql_count,result
0,Duplicate Bank Transaction,1,1,PASS
1,Matched,24,24,PASS
2,Missing from Bank Statement,1,1,PASS



OVERALL VALIDATION: PASS


In [52]:
sql_file = SQL_DIR / "bank_reconciliation.sql"

sql_script = """
-- =====================================================
-- NORTHSTAR GLOBAL SERVICES LTD.
-- November 2026 Bank Reconciliation
-- =====================================================

WITH gl_bank AS (
    SELECT
        transaction_id,
        ROUND(SUM(debit - credit), 2) AS gl_signed_amount
    FROM general_ledger
    WHERE account_code = 1000
      AND transaction_id != 'OPENING001'
    GROUP BY transaction_id
),

bank AS (
    SELECT
        transaction_id,
        COUNT(*) AS bank_record_count,
        ROUND(SUM(signed_amount), 2) AS bank_signed_amount
    FROM bank_statement
    GROUP BY transaction_id
),

all_transactions AS (
    SELECT transaction_id FROM gl_bank
    UNION
    SELECT transaction_id FROM bank
)

SELECT
    t.transaction_id,
    g.gl_signed_amount,
    b.bank_signed_amount,
    COALESCE(b.bank_record_count, 0) AS bank_record_count,

    CASE
        WHEN b.bank_record_count > 1
            THEN 'Duplicate Bank Transaction'
        WHEN g.transaction_id IS NULL
            THEN 'Not Recorded in General Ledger'
        WHEN b.transaction_id IS NULL
            THEN 'Missing from Bank Statement'
        WHEN ABS(g.gl_signed_amount - b.bank_signed_amount) > 0.01
            THEN 'Amount Mismatch'
        ELSE 'Matched'
    END AS reconciliation_status

FROM all_transactions AS t

LEFT JOIN gl_bank AS g
    ON t.transaction_id = g.transaction_id

LEFT JOIN bank AS b
    ON t.transaction_id = b.transaction_id

ORDER BY t.transaction_id;
"""

sql_file.write_text(sql_script.strip(), encoding="utf-8")

print("SQL script saved successfully:")
print(sql_file)

SQL script saved successfully:
C:\Users\hp\Desktop\Gdansk University of Technology, Poland\Job\AON\Project\finance-rtr-analytics\sql\bank_reconciliation.sql


In [53]:
print("BUDGET TABLE STRUCTURE")
print("=" * 45)

display(pd.read_sql_query(
    "PRAGMA table_info(budget)",
    connection
))

print("\nBUDGET DATA")
display(pd.read_sql_query(
    "SELECT * FROM budget LIMIT 10",
    connection
))

BUDGET TABLE STRUCTURE


,cid,name,type,notnull,dflt_value,pk
0,0,category,TEXT,0,None,0
1,1,account_name,TEXT,0,None,0
2,2,account_code,INTEGER,0,None,0
3,3,budget_amount,INTEGER,0,None,0
4,4,period,TEXT,0,None,0



BUDGET DATA


,category,account_name,account_code,budget_amount,period
0,Revenue,Service Revenue,4000,105000,2026-11
1,Revenue,Interest Income,4100,200,2026-11
2,Expense,Salaries Expense,5000,22000,2026-11
3,Expense,Rent Expense,5100,4000,2026-11
4,Expense,Utilities Expense,5200,1000,2026-11
5,Expense,Software Expense,5300,1200,2026-11
6,Expense,Travel Expense,5400,1500,2026-11
7,Expense,Depreciation Expense,5500,250,2026-11
8,Expense,Bank Charges,5600,200,2026-11
9,Expense,Interest Expense,5700,500,2026-11


In [55]:
sql_budget_analysis = """
WITH actuals AS (
    SELECT
        gl.account_code,
        ROUND(
            SUM(
                CASE
                    WHEN coa.account_type = 'Revenue'
                        THEN gl.credit - gl.debit
                    WHEN coa.account_type = 'Expense'
                        THEN gl.debit - gl.credit
                    ELSE 0
                END
            ), 2
        ) AS actual_amount

    FROM general_ledger AS gl

    INNER JOIN chart_of_accounts AS coa
        ON gl.account_code = coa.account_code

    WHERE coa.account_type IN ('Revenue', 'Expense')

    GROUP BY gl.account_code
)

SELECT
    b.account_code,
    b.account_name,
    b.category,
    b.budget_amount,
    COALESCE(a.actual_amount, 0) AS actual_amount,

    ROUND(
        CASE
            WHEN b.category = 'Revenue'
                THEN COALESCE(a.actual_amount, 0) - b.budget_amount
            ELSE b.budget_amount - COALESCE(a.actual_amount, 0)
        END, 2
    ) AS variance,

    ROUND(
        CASE
            WHEN b.budget_amount = 0 THEN NULL
            ELSE
                (
                    CASE
                        WHEN b.category = 'Revenue'
                            THEN COALESCE(a.actual_amount, 0) - b.budget_amount
                        ELSE b.budget_amount - COALESCE(a.actual_amount, 0)
                    END
                ) * 100.0 / b.budget_amount
        END, 2
    ) AS variance_percentage,

    CASE
        WHEN ABS(
            CASE
                WHEN b.category = 'Revenue'
                    THEN COALESCE(a.actual_amount, 0) - b.budget_amount
                ELSE b.budget_amount - COALESCE(a.actual_amount, 0)
            END
        ) < 0.01
            THEN 'On Budget'

        WHEN
            CASE
                WHEN b.category = 'Revenue'
                    THEN COALESCE(a.actual_amount, 0) - b.budget_amount
                ELSE b.budget_amount - COALESCE(a.actual_amount, 0)
            END > 0
            THEN 'Favourable'

        ELSE 'Unfavourable'
    END AS performance_status

FROM budget AS b

LEFT JOIN actuals AS a
    ON b.account_code = a.account_code

WHERE b.period = '2026-11'

ORDER BY
    b.category,
    b.account_code;
"""

sql_budget_vs_actual = pd.read_sql_query(
    sql_budget_analysis,
    connection
)

display(sql_budget_vs_actual)

,account_code,account_name,category,budget_amount,actual_amount,variance,variance_percentage,performance_status
0,5000,Salaries Expense,Expense,22000,30877.97,-8877.97,-40.35,Unfavourable
1,5100,Rent Expense,Expense,4000,6037.88,-2037.88,-50.95,Unfavourable
2,5200,Utilities Expense,Expense,1000,12258.58,-11258.58,-1125.86,Unfavourable
3,5300,Software Expense,Expense,1200,11925.84,-10725.84,-893.82,Unfavourable
4,5400,Travel Expense,Expense,1500,7377.40,-5877.40,-391.83,Unfavourable
5,5500,Depreciation Expense,Expense,250,250.00,0.00,0.00,On Budget
6,5600,Bank Charges,Expense,200,187.92,12.08,6.04,Favourable
7,5700,Interest Expense,Expense,500,0.00,500.00,100.00,Favourable
8,4000,Service Revenue,Revenue,105000,101413.91,-3586.09,-3.42,Unfavourable
9,4100,Interest Income,Revenue,200,150.00,-50.00,-25.00,Unfavourable


In [56]:
python_budget = pd.read_csv(
    PROCESSED_DATA / "budget_vs_actual.csv"
)

print("PYTHON BUDGET REPORT COLUMNS")
print(python_budget.columns.tolist())

print("\nSQL BUDGET REPORT COLUMNS")
print(sql_budget_vs_actual.columns.tolist())

PYTHON BUDGET REPORT COLUMNS
['period', 'category', 'account_code', 'account_name', 'budget_amount', 'actual_amount', 'variance', 'variance_percentage', 'performance']

SQL BUDGET REPORT COLUMNS
['account_code', 'account_name', 'category', 'budget_amount', 'actual_amount', 'variance', 'variance_percentage', 'performance_status']


In [57]:
# Select and rename the relevant Python columns
python_comparison = python_budget[
    [
        "account_code",
        "budget_amount",
        "actual_amount",
        "variance",
        "variance_percentage",
        "performance"
    ]
].rename(columns={
    "performance": "performance_status"
})

# Select the corresponding SQL columns
sql_comparison = sql_budget_vs_actual[
    [
        "account_code",
        "budget_amount",
        "actual_amount",
        "variance",
        "variance_percentage",
        "performance_status"
    ]
]

# Merge the two reports using account code
comparison = python_comparison.merge(
    sql_comparison,
    on="account_code",
    suffixes=("_python", "_sql"),
    how="outer",
    indicator=True
)

# Compare the numerical fields
for column in [
    "budget_amount",
    "actual_amount",
    "variance",
    "variance_percentage"
]:
    comparison[f"{column}_check"] = (
        (comparison[f"{column}_python"] -
         comparison[f"{column}_sql"]).abs() < 0.01
    )

# Compare performance classifications
comparison["performance_check"] = (
    comparison["performance_status_python"]
    == comparison["performance_status_sql"]
)

check_columns = [
    "budget_amount_check",
    "actual_amount_check",
    "variance_check",
    "variance_percentage_check",
    "performance_check"
]

comparison["overall_result"] = comparison[
    check_columns
].all(axis=1).map({
    True: "PASS",
    False: "INVESTIGATE"
})

display(comparison)

,account_code,budget_amount_python,actual_amount_python,variance_python,variance_percentage_python,performance_status_python,budget_amount_sql,actual_amount_sql,variance_sql,variance_percentage_sql,performance_status_sql,_merge,budget_amount_check,actual_amount_check,variance_check,variance_percentage_check,performance_check,overall_result
0,4000,105000,101413.91,-3586.09,-3.42,Unfavourable,105000,101413.91,-3586.09,-3.42,Unfavourable,both,True,True,True,True,True,PASS
1,4100,200,150.00,-50.00,-25.00,Unfavourable,200,150.00,-50.00,-25.00,Unfavourable,both,True,True,True,True,True,PASS
2,5000,22000,30877.97,-8877.97,-40.35,Unfavourable,22000,30877.97,-8877.97,-40.35,Unfavourable,both,True,True,True,True,True,PASS
3,5100,4000,6037.88,-2037.88,-50.95,Unfavourable,4000,6037.88,-2037.88,-50.95,Unfavourable,both,True,True,True,True,True,PASS
4,5200,1000,12258.58,-11258.58,-1125.86,Unfavourable,1000,12258.58,-11258.58,-1125.86,Unfavourable,both,True,True,True,True,True,PASS
5,5300,1200,11925.84,-10725.84,-893.82,Unfavourable,1200,11925.84,-10725.84,-893.82,Unfavourable,both,True,True,True,True,True,PASS
6,5400,1500,7377.40,-5877.40,-391.83,Unfavourable,1500,7377.40,-5877.40,-391.83,Unfavourable,both,True,True,True,True,True,PASS
7,5500,250,250.00,0.00,0.00,On Budget,250,250.00,0.00,0.00,On Budget,both,True,True,True,True,True,PASS
8,5600,200,187.92,12.08,6.04,Favourable,200,187.92,12.08,6.04,Favourable,both,True,True,True,True,True,PASS
9,5700,500,0.00,500.00,100.00,Favourable,500,0.00,500.00,100.00,Favourable,both,True,True,True,True,True,PASS


In [58]:
print("PYTHON VS SQL BUDGET VALIDATION")
print("=" * 50)

print("Accounts compared:", len(comparison))

print(
    "Accounts passing:",
    (comparison["overall_result"] == "PASS").sum()
)

print(
    "Accounts requiring investigation:",
    (comparison["overall_result"] == "INVESTIGATE").sum()
)

print(
    "\nOVERALL RESULT:",
    "PASS"
    if (comparison["overall_result"] == "PASS").all()
    else "INVESTIGATE"
)

PYTHON VS SQL BUDGET VALIDATION
Accounts compared: 10
Accounts passing: 10
Accounts requiring investigation: 0

OVERALL RESULT: PASS


In [59]:
sql_budget_file = SQL_DIR / "budget_vs_actual.sql"

sql_budget_file.write_text(
    sql_budget_analysis.strip(),
    encoding="utf-8"
)

print("Budget analysis SQL saved successfully:")
print(sql_budget_file)

Budget analysis SQL saved successfully:
C:\Users\hp\Desktop\Gdansk University of Technology, Poland\Job\AON\Project\finance-rtr-analytics\sql\budget_vs_actual.sql


In [60]:
sql_profit_comparison = """
WITH budget_totals AS (
    SELECT
        SUM(
            CASE WHEN category = 'Revenue'
                THEN budget_amount ELSE 0 END
        ) AS budget_revenue,

        SUM(
            CASE WHEN category = 'Expense'
                THEN budget_amount ELSE 0 END
        ) AS budget_expenses

    FROM budget
    WHERE period = '2026-11'
),

actual_totals AS (
    SELECT
        SUM(
            CASE WHEN coa.account_type = 'Revenue'
                THEN gl.credit - gl.debit ELSE 0 END
        ) AS actual_revenue,

        SUM(
            CASE WHEN coa.account_type = 'Expense'
                THEN gl.debit - gl.credit ELSE 0 END
        ) AS actual_expenses

    FROM general_ledger AS gl

    INNER JOIN chart_of_accounts AS coa
        ON gl.account_code = coa.account_code

    WHERE coa.account_type IN ('Revenue', 'Expense')
)

SELECT
    ROUND(budget_revenue, 2) AS budget_revenue,
    ROUND(actual_revenue, 2) AS actual_revenue,

    ROUND(budget_expenses, 2) AS budget_expenses,
    ROUND(actual_expenses, 2) AS actual_expenses,

    ROUND(budget_revenue - budget_expenses, 2)
        AS budget_net_profit,

    ROUND(actual_revenue - actual_expenses, 2)
        AS actual_net_profit,

    ROUND(
        (actual_revenue - actual_expenses)
        - (budget_revenue - budget_expenses),
        2
    ) AS net_profit_variance,

    ROUND(
        (
            (actual_revenue - actual_expenses)
            - (budget_revenue - budget_expenses)
        ) * 100.0
        / (budget_revenue - budget_expenses),
        2
    ) AS net_profit_variance_percentage

FROM budget_totals, actual_totals;
"""

sql_profit_summary = pd.read_sql_query(
    sql_profit_comparison,
    connection
)

display(sql_profit_summary)

,budget_revenue,actual_revenue,budget_expenses,actual_expenses,budget_net_profit,actual_net_profit,net_profit_variance,net_profit_variance_percentage
0,105200.0,101563.91,30650.0,68915.59,74550.0,32648.32,-41901.68,-56.21


In [61]:
sql_profit_file = SQL_DIR / "net_profit_budget_vs_actual.sql"

sql_profit_file.write_text(
    sql_profit_comparison.strip(),
    encoding="utf-8"
)

print("Net profit SQL script saved successfully:")
print(sql_profit_file)

Net profit SQL script saved successfully:
C:\Users\hp\Desktop\Gdansk University of Technology, Poland\Job\AON\Project\finance-rtr-analytics\sql\net_profit_budget_vs_actual.sql


In [62]:
print("SQL PROJECT FILES")
print("=" * 45)

for sql_path in sorted(SQL_DIR.glob("*.sql")):
    print(f"✓ {sql_path.name}")

SQL PROJECT FILES
✓ bank_reconciliation.sql
✓ budget_vs_actual.sql
✓ net_profit_budget_vs_actual.sql


In [63]:
print("MANAGEMENT KPI SUMMARY")
print("=" * 45)

management_kpis = pd.read_csv(
    PROCESSED_DATA / "management_kpi_summary.csv"
)

display(management_kpis)

print("\nMONTH-END CONTROL SUMMARY")
print("=" * 45)

control_summary = pd.read_csv(
    PROCESSED_DATA / "month_end_control_summary.csv"
)

display(control_summary)

MANAGEMENT KPI SUMMARY


,Indicator,Value
0,Total Revenue,101563.910000
1,Total Expenses,68915.590000
2,Net Profit,32648.320000
3,Net Profit Margin,32.145592
4,Revenue Budget Variance (%),-3.456359
5,Expense Budget Variance (%),124.846949
6,Net Profit Budget Variance (%),-56.206144
7,Open Reconciliation Exceptions,2.000000
8,Total Flagged Transaction Value,23121.480000
9,Journal Entries Reviewed,52.000000



MONTH-END CONTROL SUMMARY


,Control,Result,Status
0,Journal entries reviewed,52,Pass
1,Unbalanced journal entries,0,Pass
2,Unexpected journal line counts,0,Pass
3,Duplicate journal lines,0,Pass
4,Invalid account codes,0,Pass
5,Transactions outside reporting period,0,Pass
6,Open bank reconciliation exceptions,2,Open
7,High-severity open exceptions,2,Open


In [64]:
sql_management_report = """
WITH financials AS (
    SELECT
        SUM(
            CASE WHEN coa.account_type = 'Revenue'
                THEN gl.credit - gl.debit ELSE 0 END
        ) AS total_revenue,

        SUM(
            CASE WHEN coa.account_type = 'Expense'
                THEN gl.debit - gl.credit ELSE 0 END
        ) AS total_expenses

    FROM general_ledger AS gl

    INNER JOIN chart_of_accounts AS coa
        ON gl.account_code = coa.account_code

    WHERE coa.account_type IN ('Revenue', 'Expense')
),

budgets AS (
    SELECT
        SUM(CASE WHEN category = 'Revenue'
            THEN budget_amount ELSE 0 END) AS budget_revenue,

        SUM(CASE WHEN category = 'Expense'
            THEN budget_amount ELSE 0 END) AS budget_expenses

    FROM budget
    WHERE period = '2026-11'
),

gl_bank AS (
    SELECT
        transaction_id,
        ROUND(SUM(debit - credit), 2) AS gl_signed_amount
    FROM general_ledger
    WHERE account_code = 1000
      AND transaction_id != 'OPENING001'
    GROUP BY transaction_id
),

bank AS (
    SELECT
        transaction_id,
        COUNT(*) AS bank_record_count,
        ROUND(SUM(signed_amount), 2) AS bank_signed_amount
    FROM bank_statement
    GROUP BY transaction_id
),

all_transactions AS (
    SELECT transaction_id FROM gl_bank
    UNION
    SELECT transaction_id FROM bank
),

reconciliation AS (
    SELECT
        t.transaction_id,
        g.gl_signed_amount,
        b.bank_signed_amount,
        b.bank_record_count,

        CASE
            WHEN b.bank_record_count > 1
                THEN 'Duplicate Bank Transaction'
            WHEN g.transaction_id IS NULL
                THEN 'Not Recorded in General Ledger'
            WHEN b.transaction_id IS NULL
                THEN 'Missing from Bank Statement'
            WHEN ABS(g.gl_signed_amount - b.bank_signed_amount) > 0.01
                THEN 'Amount Mismatch'
            ELSE 'Matched'
        END AS reconciliation_status

    FROM all_transactions AS t

    LEFT JOIN gl_bank AS g
        ON t.transaction_id = g.transaction_id

    LEFT JOIN bank AS b
        ON t.transaction_id = b.transaction_id
),

reconciliation_kpis AS (
    SELECT
        SUM(
            CASE WHEN reconciliation_status != 'Matched'
                THEN 1 ELSE 0 END
        ) AS open_exceptions,

        SUM(
            CASE WHEN reconciliation_status != 'Matched'
                THEN
                    CASE
                        WHEN gl_signed_amount IS NOT NULL
                            THEN ABS(gl_signed_amount)
                        ELSE ABS(bank_signed_amount)
                    END
                ELSE 0
            END
        ) AS flagged_transaction_value

    FROM reconciliation
),

journal_kpis AS (
    SELECT
        COUNT(DISTINCT transaction_id) AS journal_entries_reviewed
    FROM general_ledger
),

metrics AS (
    SELECT
        f.total_revenue,
        f.total_expenses,
        f.total_revenue - f.total_expenses AS net_profit,

        (f.total_revenue - f.total_expenses)
            * 100.0 / NULLIF(f.total_revenue, 0)
            AS net_profit_margin,

        (f.total_revenue - b.budget_revenue)
            * 100.0 / NULLIF(b.budget_revenue, 0)
            AS revenue_budget_variance_pct,

        (f.total_expenses - b.budget_expenses)
            * 100.0 / NULLIF(b.budget_expenses, 0)
            AS expense_budget_variance_pct,

        (
            (f.total_revenue - f.total_expenses)
            - (b.budget_revenue - b.budget_expenses)
        ) * 100.0
            / NULLIF(
                b.budget_revenue - b.budget_expenses, 0
            ) AS net_profit_budget_variance_pct,

        r.open_exceptions,
        r.flagged_transaction_value,
        j.journal_entries_reviewed

    FROM financials AS f
    CROSS JOIN budgets AS b
    CROSS JOIN reconciliation_kpis AS r
    CROSS JOIN journal_kpis AS j
)

SELECT 'Total Revenue' AS indicator,
       ROUND(total_revenue, 2) AS value
FROM metrics

UNION ALL

SELECT 'Total Expenses',
       ROUND(total_expenses, 2)
FROM metrics

UNION ALL

SELECT 'Net Profit',
       ROUND(net_profit, 2)
FROM metrics

UNION ALL

SELECT 'Net Profit Margin',
       ROUND(net_profit_margin, 6)
FROM metrics

UNION ALL

SELECT 'Revenue Budget Variance (%)',
       ROUND(revenue_budget_variance_pct, 6)
FROM metrics

UNION ALL

SELECT 'Expense Budget Variance (%)',
       ROUND(expense_budget_variance_pct, 6)
FROM metrics

UNION ALL

SELECT 'Net Profit Budget Variance (%)',
       ROUND(net_profit_budget_variance_pct, 6)
FROM metrics

UNION ALL

SELECT 'Open Reconciliation Exceptions',
       open_exceptions
FROM metrics

UNION ALL

SELECT 'Total Flagged Transaction Value',
       ROUND(flagged_transaction_value, 2)
FROM metrics

UNION ALL

SELECT 'Journal Entries Reviewed',
       journal_entries_reviewed
FROM metrics;
"""

sql_management_kpis = pd.read_sql_query(
    sql_management_report,
    connection
)

display(sql_management_kpis)

,indicator,value
0,Total Revenue,101563.910000
1,Total Expenses,68915.590000
2,Net Profit,32648.320000
3,Net Profit Margin,32.145592
4,Revenue Budget Variance (%),-3.456359
5,Expense Budget Variance (%),124.846949
6,Net Profit Budget Variance (%),-56.206144
7,Open Reconciliation Exceptions,2.000000
8,Total Flagged Transaction Value,23121.480000
9,Journal Entries Reviewed,52.000000


In [65]:
print("CONSOLIDATED SQL MANAGEMENT REPORT")
print("=" * 50)

print("Number of management indicators:",
      len(sql_management_kpis))

print("\nManagement indicators:")
display(sql_management_kpis)

print(
    "\nReport completeness:",
    "PASS" if len(sql_management_kpis) == 10
    else "INVESTIGATE"
)

CONSOLIDATED SQL MANAGEMENT REPORT
Number of management indicators: 10

Management indicators:


,indicator,value
0,Total Revenue,101563.910000
1,Total Expenses,68915.590000
2,Net Profit,32648.320000
3,Net Profit Margin,32.145592
4,Revenue Budget Variance (%),-3.456359
5,Expense Budget Variance (%),124.846949
6,Net Profit Budget Variance (%),-56.206144
7,Open Reconciliation Exceptions,2.000000
8,Total Flagged Transaction Value,23121.480000
9,Journal Entries Reviewed,52.000000



Report completeness: PASS


In [66]:
# Prepare the original Python KPI report
python_kpi_comparison = management_kpis.copy()

python_kpi_comparison.columns = [
    "indicator",
    "python_value"
]

# Prepare the SQL KPI report
sql_kpi_comparison = sql_management_kpis.copy()

sql_kpi_comparison.columns = [
    "indicator",
    "sql_value"
]

# Compare the two reports
kpi_comparison = python_kpi_comparison.merge(
    sql_kpi_comparison,
    on="indicator",
    how="outer",
    indicator=True
)

# Check numerical agreement
kpi_comparison["difference"] = (
    kpi_comparison["python_value"]
    - kpi_comparison["sql_value"]
).abs()

kpi_comparison["result"] = (
    (kpi_comparison["_merge"] == "both")
    & (kpi_comparison["difference"] < 0.01)
).map({
    True: "PASS",
    False: "INVESTIGATE"
})

display(kpi_comparison)

,indicator,python_value,sql_value,_merge,difference,result
0,Expense Budget Variance (%),124.846949,124.846949,both,4.290375e-07,PASS
1,Journal Entries Reviewed,52.000000,52.000000,both,0.000000e+00,PASS
2,Net Profit,32648.320000,32648.320000,both,0.000000e+00,PASS
3,Net Profit Budget Variance (%),-56.206144,-56.206144,both,4.721663e-07,PASS
4,Net Profit Margin,32.145592,32.145592,both,1.258786e-07,PASS
5,Open Reconciliation Exceptions,2.000000,2.000000,both,0.000000e+00,PASS
6,Revenue Budget Variance (%),-3.456359,-3.456359,both,3.155894e-07,PASS
7,Total Expenses,68915.590000,68915.590000,both,0.000000e+00,PASS
8,Total Flagged Transaction Value,23121.480000,23121.480000,both,0.000000e+00,PASS
9,Total Revenue,101563.910000,101563.910000,both,0.000000e+00,PASS


In [67]:
print("PYTHON VS SQL MANAGEMENT KPI VALIDATION")
print("=" * 50)

print("Indicators compared:", len(kpi_comparison))

print(
    "Passed:",
    (kpi_comparison["result"] == "PASS").sum()
)

print(
    "Requiring investigation:",
    (kpi_comparison["result"] == "INVESTIGATE").sum()
)

print(
    "\nOVERALL RESULT:",
    "PASS"
    if (kpi_comparison["result"] == "PASS").all()
    else "INVESTIGATE"
)

PYTHON VS SQL MANAGEMENT KPI VALIDATION
Indicators compared: 10
Passed: 10
Requiring investigation: 0

OVERALL RESULT: PASS


In [68]:
sql_management_file = SQL_DIR / "management_kpi_report.sql"

sql_management_file.write_text(
    sql_management_report.strip(),
    encoding="utf-8"
)

print("Management KPI SQL saved successfully:")
print(sql_management_file)

Management KPI SQL saved successfully:
C:\Users\hp\Desktop\Gdansk University of Technology, Poland\Job\AON\Project\finance-rtr-analytics\sql\management_kpi_report.sql


In [69]:
print("SQL PROJECT FILES")
print("=" * 50)

for sql_path in sorted(SQL_DIR.glob("*.sql")):
    print("✓", sql_path.name)

print("\nTotal SQL scripts:", len(list(SQL_DIR.glob("*.sql"))))

SQL PROJECT FILES
✓ bank_reconciliation.sql
✓ budget_vs_actual.sql
✓ management_kpi_report.sql
✓ net_profit_budget_vs_actual.sql

Total SQL scripts: 4


In [70]:
sql_month_end_controls = """
WITH journal_checks AS (
    SELECT
        transaction_id,
        COUNT(*) AS line_count,
        ROUND(SUM(debit), 2) AS total_debit,
        ROUND(SUM(credit), 2) AS total_credit
    FROM general_ledger
    GROUP BY transaction_id
),

gl_checks AS (
    SELECT
        COUNT(DISTINCT gl.transaction_id) AS journal_count,

        SUM(
            CASE
                WHEN coa.account_code IS NULL THEN 1
                ELSE 0
            END
        ) AS invalid_account_records,

        SUM(
            CASE
                WHEN date(gl.date) < '2026-11-01'
                  OR date(gl.date) >= '2026-12-01'
                THEN 1
                ELSE 0
            END
        ) AS outside_period_records

    FROM general_ledger AS gl
    LEFT JOIN chart_of_accounts AS coa
        ON gl.account_code = coa.account_code
),

duplicate_lines AS (
    SELECT
        COALESCE(SUM(duplicate_count - 1), 0) AS duplicate_line_count
    FROM (
        SELECT COUNT(*) AS duplicate_count
        FROM general_ledger
        GROUP BY
            transaction_id,
            date,
            description,
            department,
            account_code,
            debit,
            credit
        HAVING COUNT(*) > 1
    )
),

reconciliation AS (
    SELECT
        t.transaction_id,

        CASE
            WHEN b.bank_record_count > 1
                THEN 'Duplicate Bank Transaction'
            WHEN g.transaction_id IS NULL
                THEN 'Not Recorded in General Ledger'
            WHEN b.transaction_id IS NULL
                THEN 'Missing from Bank Statement'
            WHEN ABS(g.gl_signed_amount - b.bank_signed_amount) > 0.01
                THEN 'Amount Mismatch'
            ELSE 'Matched'
        END AS reconciliation_status

    FROM (
        SELECT transaction_id FROM (
            SELECT DISTINCT transaction_id
            FROM general_ledger
            WHERE account_code = 1000
              AND transaction_id != 'OPENING001'
            UNION
            SELECT DISTINCT transaction_id
            FROM bank_statement
        )
    ) AS t

    LEFT JOIN (
        SELECT
            transaction_id,
            SUM(debit - credit) AS gl_signed_amount
        FROM general_ledger
        WHERE account_code = 1000
          AND transaction_id != 'OPENING001'
        GROUP BY transaction_id
    ) AS g
        ON t.transaction_id = g.transaction_id

    LEFT JOIN (
        SELECT
            transaction_id,
            COUNT(*) AS bank_record_count,
            SUM(signed_amount) AS bank_signed_amount
        FROM bank_statement
        GROUP BY transaction_id
    ) AS b
        ON t.transaction_id = b.transaction_id
),

control_values AS (
    SELECT
        gc.journal_count,

        (
            SELECT COUNT(*)
            FROM journal_checks
            WHERE ABS(total_debit - total_credit) > 0.01
        ) AS unbalanced_journals,

        (
            SELECT COUNT(*)
            FROM journal_checks
            WHERE line_count != 2
        ) AS unexpected_line_counts,

        dl.duplicate_line_count,
        gc.invalid_account_records,
        gc.outside_period_records,

        (
            SELECT COUNT(*)
            FROM reconciliation
            WHERE reconciliation_status != 'Matched'
        ) AS open_reconciliation_exceptions

    FROM gl_checks AS gc
    CROSS JOIN duplicate_lines AS dl
)

SELECT
    'Journal entries reviewed' AS control,
    journal_count AS result,
    'Pass' AS status
FROM control_values

UNION ALL

SELECT
    'Unbalanced journal entries',
    unbalanced_journals,
    CASE WHEN unbalanced_journals = 0
        THEN 'Pass' ELSE 'Investigate' END
FROM control_values

UNION ALL

SELECT
    'Unexpected journal line counts',
    unexpected_line_counts,
    CASE WHEN unexpected_line_counts = 0
        THEN 'Pass' ELSE 'Investigate' END
FROM control_values

UNION ALL

SELECT
    'Duplicate journal lines',
    duplicate_line_count,
    CASE WHEN duplicate_line_count = 0
        THEN 'Pass' ELSE 'Investigate' END
FROM control_values

UNION ALL

SELECT
    'Invalid account codes',
    invalid_account_records,
    CASE WHEN invalid_account_records = 0
        THEN 'Pass' ELSE 'Investigate' END
FROM control_values

UNION ALL

SELECT
    'Transactions outside reporting period',
    outside_period_records,
    CASE WHEN outside_period_records = 0
        THEN 'Pass' ELSE 'Investigate' END
FROM control_values

UNION ALL

SELECT
    'Open bank reconciliation exceptions',
    open_reconciliation_exceptions,
    CASE WHEN open_reconciliation_exceptions = 0
        THEN 'Pass' ELSE 'Open' END
FROM control_values;
"""

sql_control_summary = pd.read_sql_query(
    sql_month_end_controls,
    connection
)

display(sql_control_summary)

,control,result,status
0,Journal entries reviewed,52,Pass
1,Unbalanced journal entries,0,Pass
2,Unexpected journal line counts,0,Pass
3,Duplicate journal lines,0,Pass
4,Invalid account codes,0,Pass
5,Transactions outside reporting period,0,Pass
6,Open bank reconciliation exceptions,2,Open


In [71]:
print("SQL MONTH-END CONTROL REPORT")
print("=" * 50)

print("Total controls:", len(sql_control_summary))

print("\nControl status summary:")
display(
    sql_control_summary["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="count")
)

print(
    "\nControls passed:",
    (sql_control_summary["status"] == "Pass").sum()
)

print(
    "Controls requiring investigation:",
    (sql_control_summary["status"] == "Investigate").sum()
)

print(
    "Open controls:",
    (sql_control_summary["status"] == "Open").sum()
)

SQL MONTH-END CONTROL REPORT
Total controls: 7

Control status summary:


,status,count
0,Pass,6
1,Open,1



Controls passed: 6
Controls requiring investigation: 0
Open controls: 1


In [72]:
python_control_comparison = control_summary[
    ["Control", "Result", "Status"]
].copy()

sql_control_comparison = sql_control_summary[
    ["control", "result", "status"]
].copy()

# Standardise column names
python_control_comparison.columns = [
    "control", "python_result", "python_status"
]

sql_control_comparison.columns = [
    "control", "sql_result", "sql_status"
]

control_comparison = python_control_comparison.merge(
    sql_control_comparison,
    on="control",
    how="outer",
    indicator=True
)

control_comparison["result_check"] = (
    control_comparison["python_result"]
    == control_comparison["sql_result"]
)

control_comparison["status_check"] = (
    control_comparison["python_status"]
    == control_comparison["sql_status"]
)

control_comparison["validation"] = (
    control_comparison["result_check"]
    & control_comparison["status_check"]
    & (control_comparison["_merge"] == "both")
).map({
    True: "PASS",
    False: "INVESTIGATE"
})

display(control_comparison)

,control,python_result,python_status,sql_result,sql_status,_merge,result_check,status_check,validation
0,Duplicate journal lines,0,Pass,0.0,Pass,both,True,True,PASS
1,High-severity open exceptions,2,Open,NaN,NaN,left_only,False,False,INVESTIGATE
2,Invalid account codes,0,Pass,0.0,Pass,both,True,True,PASS
3,Journal entries reviewed,52,Pass,52.0,Pass,both,True,True,PASS
4,Open bank reconciliation exceptions,2,Open,2.0,Open,both,True,True,PASS
5,Transactions outside reporting period,0,Pass,0.0,Pass,both,True,True,PASS
6,Unbalanced journal entries,0,Pass,0.0,Pass,both,True,True,PASS
7,Unexpected journal line counts,0,Pass,0.0,Pass,both,True,True,PASS


In [73]:
print("PYTHON VS SQL CONTROL VALIDATION")
print("=" * 50)

print("Controls compared:", len(control_comparison))

print(
    "Passed:",
    (control_comparison["validation"] == "PASS").sum()
)

print(
    "Requiring investigation:",
    (control_comparison["validation"] == "INVESTIGATE").sum()
)

print(
    "\nOVERALL RESULT:",
    "PASS"
    if (control_comparison["validation"] == "PASS").all()
    else "INVESTIGATE"
)

PYTHON VS SQL CONTROL VALIDATION
Controls compared: 8
Passed: 7
Requiring investigation: 1

OVERALL RESULT: INVESTIGATE


In [74]:
exception_register = pd.read_csv(
    PROCESSED_DATA / "exception_register.csv"
)

print("Columns:")
print(exception_register.columns.tolist())

display(exception_register)

Columns:
['exception_id', 'transaction_id', 'exception_type', 'amount', 'severity', 'status', 'recommended_action']


,exception_id,transaction_id,exception_type,amount,severity,status,recommended_action
0,EXC001,TXN00015,Missing from Bank Statement,11780.60,High,Open,Investigate bank records and confirm whether t...
1,EXC002,TXN00018,Duplicate Bank Transaction,11340.88,High,Open,Verify the duplicate entry and confirm whether...


In [75]:
exception_register.to_sql(
    "exception_register",
    connection,
    if_exists="replace",
    index=False
)

# Verify the imported table
pd.read_sql_query(
    "SELECT * FROM exception_register",
    connection
)

,exception_id,transaction_id,exception_type,amount,severity,status,recommended_action
0,EXC001,TXN00015,Missing from Bank Statement,11780.60,High,Open,Investigate bank records and confirm whether t...
1,EXC002,TXN00018,Duplicate Bank Transaction,11340.88,High,Open,Verify the duplicate entry and confirm whether...


In [77]:
exception_register.to_sql(
    "exception_register",
    connection,
    if_exists="replace",
    index=False
)

2

In [78]:

control_sql = """
WITH journal_summary AS (
    SELECT
        transaction_id,
        SUM(debit) AS total_debit,
        SUM(credit) AS total_credit,
        COUNT(*) AS line_count
    FROM general_ledger
    GROUP BY transaction_id
),

unbalanced_journals AS (
    SELECT COUNT(*) AS result
    FROM journal_summary
    WHERE ABS(total_debit - total_credit) > 0.01
),

unexpected_line_counts AS (
    SELECT COUNT(*) AS result
    FROM journal_summary
    WHERE line_count != 2
),

duplicate_gl_lines AS (
    SELECT COALESCE(SUM(duplicate_count - 1), 0) AS result
    FROM (
        SELECT COUNT(*) AS duplicate_count
        FROM general_ledger
        GROUP BY transaction_id, date, description,
                 department, account_code, debit, credit
        HAVING COUNT(*) > 1
    )
),

invalid_accounts AS (
    SELECT COUNT(*) AS result
    FROM general_ledger gl
    LEFT JOIN chart_of_accounts coa
        ON gl.account_code = coa.account_code
    WHERE coa.account_code IS NULL
),

outside_period AS (
    SELECT COUNT(*) AS result
    FROM general_ledger
    WHERE date < '2026-11-01'
       OR date >= '2026-12-01'
),

gl_bank AS (
    SELECT
        transaction_id,
        SUM(debit - credit) AS gl_amount
    FROM general_ledger
    WHERE account_code = 1000
      AND transaction_id != 'OPENING001'
    GROUP BY transaction_id
),

bank_grouped AS (
    SELECT
        transaction_id,
        COUNT(*) AS bank_count,
        SUM(signed_amount) AS bank_amount
    FROM bank_statement
    GROUP BY transaction_id
),

reconciliation_status AS (
    SELECT
        CASE
            WHEN b.bank_count > 1
                THEN 'Duplicate Bank Transaction'
            WHEN b.transaction_id IS NULL
                THEN 'Missing from Bank Statement'
            WHEN g.transaction_id IS NULL
                THEN 'Not Recorded in General Ledger'
            WHEN ABS(g.gl_amount - b.bank_amount) > 0.01
                THEN 'Amount Mismatch'
            ELSE 'Matched'
        END AS reconciliation_status
    FROM gl_bank g
    LEFT JOIN bank_grouped b
        ON g.transaction_id = b.transaction_id

    UNION ALL

    SELECT
        'Not Recorded in General Ledger' AS reconciliation_status
    FROM bank_grouped b
    LEFT JOIN gl_bank g
        ON b.transaction_id = g.transaction_id
    WHERE g.transaction_id IS NULL
),

open_reconciliation_exceptions AS (
    SELECT COUNT(*) AS result
    FROM reconciliation_status
    WHERE reconciliation_status != 'Matched'
),

high_severity_exceptions AS (
    SELECT COUNT(*) AS result
    FROM exception_register
    WHERE severity = 'High'
      AND status = 'Open'
),

journal_count AS (
    SELECT COUNT(DISTINCT transaction_id) AS result
    FROM general_ledger
)

SELECT
    'Journal entries reviewed' AS control,
    result,
    'Pass' AS status
FROM journal_count

UNION ALL

SELECT
    'Unbalanced journal entries',
    result,
    CASE WHEN result = 0 THEN 'Pass' ELSE 'Investigate' END
FROM unbalanced_journals

UNION ALL

SELECT
    'Unexpected journal line counts',
    result,
    CASE WHEN result = 0 THEN 'Pass' ELSE 'Investigate' END
FROM unexpected_line_counts

UNION ALL

SELECT
    'Duplicate journal lines',
    result,
    CASE WHEN result = 0 THEN 'Pass' ELSE 'Investigate' END
FROM duplicate_gl_lines

UNION ALL

SELECT
    'Invalid account codes',
    result,
    CASE WHEN result = 0 THEN 'Pass' ELSE 'Investigate' END
FROM invalid_accounts

UNION ALL

SELECT
    'Transactions outside reporting period',
    result,
    CASE WHEN result = 0 THEN 'Pass' ELSE 'Investigate' END
FROM outside_period

UNION ALL

SELECT
    'Open bank reconciliation exceptions',
    result,
    CASE WHEN result = 0 THEN 'Pass' ELSE 'Open' END
FROM open_reconciliation_exceptions

UNION ALL

SELECT
    'High-severity open exceptions',
    result,
    CASE WHEN result = 0 THEN 'Pass' ELSE 'Open' END
FROM high_severity_exceptions
"""

sql_control_summary = pd.read_sql_query(
    control_sql,
    connection
)

display(sql_control_summary)


,control,result,status
0,Journal entries reviewed,52,Pass
1,Unbalanced journal entries,0,Pass
2,Unexpected journal line counts,0,Pass
3,Duplicate journal lines,0,Pass
4,Invalid account codes,0,Pass
5,Transactions outside reporting period,0,Pass
6,Open bank reconciliation exceptions,2,Open
7,High-severity open exceptions,2,Open


In [79]:
# Prepare Python and SQL control reports
python_control_comparison = control_summary[
    ["Control", "Result", "Status"]
].copy()

sql_control_comparison = sql_control_summary[
    ["control", "result", "status"]
].copy()

# Standardise column names
python_control_comparison.columns = [
    "control", "python_result", "python_status"
]

sql_control_comparison.columns = [
    "control", "sql_result", "sql_status"
]

# Merge both reports
control_validation = python_control_comparison.merge(
    sql_control_comparison,
    on="control",
    how="outer",
    indicator=True
)

# Compare results and statuses
control_validation["result_check"] = (
    control_validation["python_result"]
    == control_validation["sql_result"]
)

control_validation["status_check"] = (
    control_validation["python_status"]
    == control_validation["sql_status"]
)

control_validation["validation"] = control_validation.apply(
    lambda row: "PASS"
    if row["_merge"] == "both"
    and row["result_check"]
    and row["status_check"]
    else "INVESTIGATE",
    axis=1
)

display(control_validation)

# Validation summary
print("\nPYTHON VS SQL CONTROL VALIDATION")
print("=" * 50)
print(f"Controls compared: {len(control_validation)}")
print(
    f"Passed: "
    f"{(control_validation['validation'] == 'PASS').sum()}"
)
print(
    f"Requiring investigation: "
    f"{(control_validation['validation'] == 'INVESTIGATE').sum()}"
)

overall_result = (
    "PASS"
    if (control_validation["validation"] == "PASS").all()
    else "INVESTIGATE"
)

print(f"\nOVERALL RESULT: {overall_result}")

,control,python_result,python_status,sql_result,sql_status,_merge,result_check,status_check,validation
0,Duplicate journal lines,0,Pass,0,Pass,both,True,True,PASS
1,High-severity open exceptions,2,Open,2,Open,both,True,True,PASS
2,Invalid account codes,0,Pass,0,Pass,both,True,True,PASS
3,Journal entries reviewed,52,Pass,52,Pass,both,True,True,PASS
4,Open bank reconciliation exceptions,2,Open,2,Open,both,True,True,PASS
5,Transactions outside reporting period,0,Pass,0,Pass,both,True,True,PASS
6,Unbalanced journal entries,0,Pass,0,Pass,both,True,True,PASS
7,Unexpected journal line counts,0,Pass,0,Pass,both,True,True,PASS



PYTHON VS SQL CONTROL VALIDATION
Controls compared: 8
Passed: 8
Requiring investigation: 0

OVERALL RESULT: PASS


In [80]:
sql_file = SQL_DIR / "month_end_control_report.sql"

# Extract SQL query from the Python variable
sql_file.write_text(control_sql, encoding="utf-8")

print("SQL report saved successfully:")
print(sql_file)

SQL report saved successfully:
C:\Users\hp\Desktop\Gdansk University of Technology, Poland\Job\AON\Project\finance-rtr-analytics\sql\month_end_control_report.sql


In [81]:
for folder in PROJECT_ROOT.iterdir():
    print(folder.name, "/" if folder.is_dir() else "")

    if folder.is_dir():
        for item in sorted(folder.iterdir()):
            print("   ", item.name)

dashboard /
data /
    finance_rtr.db
    processed
    raw
notebooks /
    .ipynb_checkpoints
    01_project_setup.ipynb
    02_generate_financial_data.ipynb
    02_generate_financial_data.ipynb.ipynb
    03_month_end_analysis.ipynb
    04_sql_financial_reporting.ipynb
reports /
    bank_reconciliation_exceptions.png
    november_2026_management_report.md
    november_2026_management_report.pdf
    november_2026_management_report_recovered.pdf
    pdf_preview_page_1.png
    pdf_preview_page_2.png
    pdf_preview_page_3.png
    pdf_preview_page_4.png
    recovered_preview_1.png
    recovered_preview_2.png
    recovered_preview_3.png
sql /
    bank_reconciliation.sql
    budget_vs_actual.sql
    management_kpi_report.sql
    month_end_control_report.sql
    net_profit_budget_vs_actual.sql
src /


In [82]:
from pathlib import Path

files_to_check = [
    PROJECT_ROOT / "notebooks" / "02_generate_financial_data.ipynb",
    PROJECT_ROOT / "notebooks" / "02_generate_financial_data.ipynb.ipynb",
    PROJECT_ROOT / "reports" / "november_2026_management_report.pdf",
    PROJECT_ROOT / "reports" / "november_2026_management_report_recovered.pdf"
]

for file in files_to_check:
    if file.exists():
        print(f"\nFile: {file.name}")
        print(f"Size: {file.stat().st_size:,} bytes")
    else:
        print(f"\nMissing: {file.name}")


File: 02_generate_financial_data.ipynb
Size: 278,616 bytes

File: 02_generate_financial_data.ipynb.ipynb
Size: 279,597 bytes

File: november_2026_management_report.pdf
Size: 1,017 bytes

File: november_2026_management_report_recovered.pdf
Size: 8,810 bytes


In [83]:
import hashlib

notebook_1 = PROJECT_ROOT / "notebooks" / "02_generate_financial_data.ipynb"
notebook_2 = PROJECT_ROOT / "notebooks" / "02_generate_financial_data.ipynb.ipynb"

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

print("Notebook 1:", file_hash(notebook_1))
print("Notebook 2:", file_hash(notebook_2))

print("\nAre the files identical?",
      file_hash(notebook_1) == file_hash(notebook_2))

Notebook 1: e49cd22fdda67d0a0ef6cea217d8b4692866c26bac3fed6ec7fbc34af57cf892
Notebook 2: ca3e30bf3188fbb9edfa11f6113f8f3f59a86418bfe68bbcf5c6b273838e32d3

Are the files identical? False


In [84]:
import json

with open(notebook_1, "r", encoding="utf-8") as f:
    nb1 = json.load(f)

with open(notebook_2, "r", encoding="utf-8") as f:
    nb2 = json.load(f)

cells1 = nb1["cells"]
cells2 = nb2["cells"]

print("Notebook 1 cells:", len(cells1))
print("Notebook 2 cells:", len(cells2))

max_cells = max(len(cells1), len(cells2))

for i in range(max_cells):
    source1 = "".join(cells1[i]["source"]) if i < len(cells1) else "[MISSING]"
    source2 = "".join(cells2[i]["source"]) if i < len(cells2) else "[MISSING]"

    if source1 != source2:
        print(f"\n{'='*60}")
        print(f"DIFFERENCE IN CELL {i}")
        print(f"{'='*60}")
        print("\n--- NOTEBOOK 1 ---")
        print(source1[:1500])
        print("\n--- NOTEBOOK 2 ---")
        print(source2[:1500])

Notebook 1 cells: 39
Notebook 2 cells: 40

DIFFERENCE IN CELL 38

--- NOTEBOOK 1 ---


--- NOTEBOOK 2 ---
# ==========================================
# RENAME NOTEBOOK FILE
# ==========================================

notebook_folder = PROJECT_ROOT / "notebooks"

old_name = notebook_folder / "02_generate_financial_data.ipynb.ipynb"
new_name = notebook_folder / "02_generate_financial_data.ipynb"

if old_name.exists():
    old_name.rename(new_name)
    print("Notebook renamed successfully.")
elif new_name.exists():
    print("Notebook already has the correct name.")
else:
    print("Notebook file not found. Check the filename.")

DIFFERENCE IN CELL 39

--- NOTEBOOK 1 ---
[MISSING]

--- NOTEBOOK 2 ---



In [88]:
import shutil

recovered_pdf = (
    PROJECT_ROOT / "reports" /
    "november_2026_management_report_recovered.pdf"
)

final_pdf = (
    PROJECT_ROOT / "reports" /
    "November_2026_Management_Report_FINAL.pdf"
)

if recovered_pdf.exists():
    shutil.copy2(recovered_pdf, final_pdf)
    print("Final report copied successfully.")
    print("File size:", f"{final_pdf.stat().st_size:,} bytes")
else:
    print("Recovered PDF not found.")

Final report copied successfully.
File size: 8,810 bytes


In [89]:
print("Final PDF exists:", final_pdf.exists())
print("Final PDF size:", final_pdf.stat().st_size, "bytes")

Final PDF exists: True
Final PDF size: 8810 bytes


In [90]:
reports_folder = PROJECT_ROOT / "reports"

preview_files = [
    *reports_folder.glob("pdf_preview_page_*.png"),
    *reports_folder.glob("recovered_preview_*.png")
]

for file in preview_files:
    file.unlink()
    print("Removed:", file.name)

print(f"\nTotal temporary previews removed: {len(preview_files)}")

Removed: pdf_preview_page_1.png
Removed: pdf_preview_page_2.png
Removed: pdf_preview_page_3.png
Removed: pdf_preview_page_4.png
Removed: recovered_preview_1.png
Removed: recovered_preview_2.png
Removed: recovered_preview_3.png

Total temporary previews removed: 7


In [91]:
for item in sorted(reports_folder.iterdir()):
    print(item.name, "-", f"{item.stat().st_size:,} bytes")

bank_reconciliation_exceptions.png - 114,457 bytes
november_2026_management_report.md - 6,296 bytes
november_2026_management_report.pdf - 1,017 bytes
November_2026_Management_Report_FINAL.pdf - 8,810 bytes
november_2026_management_report_recovered.pdf - 8,810 bytes


In [92]:
import hashlib

pdf_final = reports_folder / "November_2026_Management_Report_FINAL.pdf"
pdf_recovered = reports_folder / "november_2026_management_report_recovered.pdf"

def get_hash(file_path):
    return hashlib.sha256(file_path.read_bytes()).hexdigest()

print("Final PDF:", get_hash(pdf_final))
print("Recovered PDF:", get_hash(pdf_recovered))

print(
    "\nAre they identical?",
    get_hash(pdf_final) == get_hash(pdf_recovered)
)

Final PDF: 7bee10361b3b7c35d5c4c7dce953b7db6b0633861cb8f4dbdb6941a9d5c9f25f
Recovered PDF: 7bee10361b3b7c35d5c4c7dce953b7db6b0633861cb8f4dbdb6941a9d5c9f25f

Are they identical? True


In [93]:
pdf_recovered.unlink()

print("Redundant PDF removed:", not pdf_recovered.exists())
print("Official PDF retained:", pdf_final.exists())

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\hp\\Desktop\\Gdansk University of Technology, Poland\\Job\\AON\\Project\\finance-rtr-analytics\\reports\\november_2026_management_report_recovered.pdf'

In [94]:

readme_content = r"""# Month-End Financial Close & Reconciliation Analytics System

## Project Overview

This project demonstrates a practical, data-driven approach to the month-end Record-to-Report (RtR) process for a fictional company, Northstar Global Services Ltd.

The project covers financial data preparation, general ledger validation, bank reconciliation, financial statement preparation, budget-versus-actual analysis, internal control monitoring, and management reporting for November 2026.

It combines Python and SQL to demonstrate how financial operations can be supported through data analysis, automated checks, and structured reporting.

> **Data disclaimer:** Northstar Global Services Ltd. is fictional. All financial records and transactions in this project are synthetically generated for educational and portfolio demonstration purposes.

## Business Objectives

The project was designed to demonstrate the ability to:

- Validate general ledger transactions and journal entries.
- Reconcile bank statement transactions against general ledger records.
- Identify, classify, and document reconciliation exceptions.
- Prepare a Profit and Loss Statement and Balance Sheet.
- Compare actual financial performance against budget.
- Monitor month-end financial controls.
- Produce management reporting and an audit trail.
- Validate consistency between Python-based and SQL-based financial reporting.

## Technology Stack

| Technology | Application |
|---|---|
| Python | Financial data processing and automation |
| pandas | Data manipulation and financial reporting |
| NumPy | Numerical operations |
| Matplotlib | Financial exception visualisation |
| SQLite | Relational financial data storage |
| SQL | Financial reporting and control queries |
| Jupyter Notebook | Reproducible analysis and documentation |
| Git / GitHub | Version control and project presentation |

## Project Workflow

### 1. Financial Data Preparation

Synthetic datasets were generated for:

- General ledger
- Chart of accounts
- Bank statement
- Accounts payable
- Accounts receivable
- Budget

### 2. General Ledger Validation

Journal entries were checked for:

- Debit and credit balancing
- Expected journal line counts
- Duplicate journal lines
- Valid account codes
- Transactions within the reporting period

### 3. Bank Reconciliation

Bank statement transactions were matched against general ledger bank-account movements.

The reconciliation process identified:

- Transactions missing from the bank statement
- Duplicate bank transactions
- Amount mismatches
- Transactions not recorded in the general ledger

An unrecorded bank service fee was also identified and posted through a simulated adjustment journal.

### 4. Financial Statement Preparation

The project generated:

- Profit and Loss Statement
- Balance Sheet
- Financial summary

The Balance Sheet was checked against the accounting equation:

**Assets = Liabilities + Equity**

### 5. Budget vs Actual Analysis

Actual revenue and expenses were compared with the approved synthetic budget.

The analysis includes:

- Budget amounts
- Actual amounts
- Variances
- Variance percentages
- Favourable and unfavourable classifications

### 6. Internal Controls and Audit Trail

Month-end controls were designed to monitor journal integrity, account validity, reporting-period accuracy, and outstanding bank reconciliation exceptions.

An exception register and audit trail were created to document identified issues and simulated resolution actions.

### 7. SQL Financial Reporting

The raw financial datasets were loaded into SQLite.

SQL queries independently reproduced key financial reports, including:

- Account balances
- Net profit
- Bank reconciliation
- Budget vs actual
- Management KPIs
- Month-end control summary

Python and SQL outputs were compared to verify consistency.

## Key Financial Results

**Reporting period:** November 2026  
**Reporting currency:** EUR

| Financial Indicator | Result |
|---|---:|
| Total Revenue | €101,563.91 |
| Total Expenses | €68,915.59 |
| Net Profit | €32,648.32 |
| Net Profit Margin | 32.15% |
| Total Assets | €117,497.99 |
| Total Liabilities | €34,849.67 |
| Total Equity | €82,648.32 |

The Balance Sheet reconciled, with a difference of €0.00.

## Reconciliation Findings

The bank reconciliation identified two outstanding exceptions:

| Exception | Transaction | Amount | Severity | Status |
|---|---|---:|---|---|
| Missing from Bank Statement | TXN00015 | €11,780.60 | High | Open |
| Duplicate Bank Transaction | TXN00018 | €11,340.88 | High | Open |

The total flagged transaction value was €23,121.48. This represents the value of flagged records, not a confirmed financial loss.

The bank service fee of €45 was recorded through a simulated adjustment journal and marked as resolved in the audit trail.

## Internal Control Results

The month-end control review covered eight controls:

| Control | Result |
|---|---:|
| Journal entries reviewed | 52 |
| Unbalanced journal entries | 0 |
| Unexpected journal line counts | 0 |
| Duplicate journal lines | 0 |
| Invalid account codes | 0 |
| Transactions outside reporting period | 0 |
| Open bank reconciliation exceptions | 2 |
| High-severity open exceptions | 2 |

The two outstanding reconciliation exceptions remain open for investigation.

## Python vs SQL Validation

Independent Python and SQL outputs were compared across the eight month-end controls.

- Controls compared: 8
- Passed: 8
- Requiring investigation due to Python/SQL disagreement: 0

**Overall validation result: PASS**

This confirms consistency between the two reporting implementations. It does not mean that all underlying financial exceptions have been resolved.

## Management Reporting

The project produces a management report in PDF and Markdown formats, summarising financial performance, reconciliation findings, internal controls, and recommended management actions.

See the `reports/` directory.

## Project Structure

```text
finance-rtr-analytics/
│
├── data/
│   ├── raw/
│   └── processed/
│
├── notebooks/
│   ├── 01_project_setup.ipynb
│   ├── 02_generate_financial_data.ipynb
│   ├── 03_month_end_analysis.ipynb
│   └── 04_sql_financial_reporting.ipynb
│
├── sql/
│   ├── bank_reconciliation.sql
│   ├── budget_vs_actual.sql
│   ├── management_kpi_report.sql
│   ├── month_end_control_report.sql
│   └── net_profit_budget_vs_actual.sql
│
├── reports/
│   ├── bank_reconciliation_exceptions.png
│   ├── november_2026_management_report.md
│   └── November_2026_Management_Report_FINAL.pdf
│
├── src/
├── dashboard/
├── README.md
└── .gitignore
```

## How to Run the Project

1. Clone or download the repository.
2. Install the required Python packages:

```bash
pip install pandas numpy matplotlib jupyter
```

3. Open the project in Jupyter Notebook.
4. Run the notebooks in numerical order, beginning with `01_project_setup.ipynb`.
5. The SQL reporting notebook uses Python's built-in `sqlite3` module to create and query the SQLite database.

The project uses synthetic data, so no external financial data connection is required.

## Limitations

- All company and transaction records are fictional.
- Reconciliation exceptions are simulated and require investigation; they do not establish actual fraud or financial loss.
- Audit-trail actions demonstrate a simulated workflow and are not external evidence of completed approvals.
- The project is a portfolio demonstration, not a production accounting system.
- Financial reporting and control logic are designed for the stated synthetic dataset and reporting period.

## Professional Relevance

This project demonstrates practical skills relevant to Record-to-Report and finance operations, including:

- General ledger analysis
- Bank reconciliation
- Financial data validation
- Month-end close support
- Budget monitoring
- Exception management
- Internal control reporting
- SQL-based financial reporting
- Management reporting

It demonstrates how Python and SQL can support accurate, traceable, and repeatable financial reporting processes.
"""

readme_path = PROJECT_ROOT / "README.md"
readme_path.write_text(readme_content, encoding="utf-8")

print("README created successfully:")
print(readme_path)
print("Characters:", len(readme_content))


README created successfully:
C:\Users\hp\Desktop\Gdansk University of Technology, Poland\Job\AON\Project\finance-rtr-analytics\README.md
Characters: 8062


In [95]:

gitignore_content = """
# Python
__pycache__/
*.py[cod]
*.pyo

# Jupyter Notebook
.ipynb_checkpoints/

# Virtual environments
.venv/
venv/
env/

# Environment variables and secrets
.env
.env.*

# Operating system files
.DS_Store
Thumbs.db
desktop.ini

# Temporary files
*.tmp
*.bak
~$*

# Obsolete and duplicate PDF reports
reports/november_2026_management_report.pdf
reports/november_2026_management_report_recovered.pdf
"""

gitignore_path = PROJECT_ROOT / ".gitignore"
gitignore_path.write_text(gitignore_content.strip() + "\n", encoding="utf-8")

print(".gitignore created successfully:")
print(gitignore_path)


.gitignore created successfully:
C:\Users\hp\Desktop\Gdansk University of Technology, Poland\Job\AON\Project\finance-rtr-analytics\.gitignore


In [96]:
for item in sorted(PROJECT_ROOT.iterdir()):
    print(
        item.name,
        "/" if item.is_dir() else ""
    )

.gitignore 
dashboard /
data /
notebooks /
README.md 
reports /
sql /
src /


In [97]:
import subprocess

try:
    result = subprocess.run(
        ["git", "--version"],
        capture_output=True,
        text=True,
        check=True
    )
    print("Git is installed.")
    print(result.stdout.strip())

except FileNotFoundError:
    print("Git is not installed or is not available in PATH.")

Git is not installed or is not available in PATH.
